# Hello

## 1. Load Data

In [ ]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9" # replace with your dataset path

In [ ]:
# Dataset Metadata
import pandas as pd

metadata_df = pd.read_csv(DATA_DOWNLOAD_DIR / "emg2pose_metadata.csv")
metadata_df.head(5)

In [ ]:
# Display training sessions
import glob
import os

sessions = sorted(glob.glob(os.path.join(DATA_DOWNLOAD_DIR, "emg2pose_dataset_mini/*.hdf5")))
sessions

## 2. Select Session To Evaluate On

In [ ]:
from emg2pose.data import Emg2PoseSessionData

# Select our dataset 
session = sessions[1]
print(session)
data = Emg2PoseSessionData(hdf5_path=session)

In [ ]:
# View the data
print(data.fields)
print(f"{'emg shape: ':<20} {data['emg'].shape}")
print(f"{'joint_angles shape: ':<20} {data['joint_angles'].shape}")
print(f"{'time shape: ':<20} {data['time'].shape}")

In [ ]:
# Show our sessions metadata

metadata_df[metadata_df["filename"] == data.metadata["filename"]]

In [ ]:
# Downscale ground-truth joint angles and display it over time

import emg2pose.visualization as visualization
from emg2pose.utils import downsample
import numpy as np

joint_angles = data["joint_angles"]
joint_angles_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250])

In [ ]:
# Show where inverse kinematics tracking (ground truth labeling)
visualization.ik_failure_plot(data)

## 3. Generate Predictions Of This Session
Load the EMG data in to the models

### Select Session(s) to Train On
Certain methods require training our own model using various ML methods.
We use our defined `sessions` list that are avaialbe to train on
and set start and end indeces to decide what subset of `sessions` to train on.

In [ ]:
print(sessions)

# --- parameters ---
start_idx = 0
end_idx = 0  # inclusive

train = sessions[start_idx:end_idx+1]
train

### 3a. Load Checkpoint and Predict
Loads a large pretrained model from Meta Reality labs and runs it on the data

In [ ]:
from pathlib import Path
import os

checkpoint_dir = Path(DATA_DOWNLOAD_DIR) / "emg2pose_model_checkpoints"

# Download checkpoint if it does not exist
if not checkpoint_dir.exists():
    os.system(f'''
    cd {DATA_DOWNLOAD_DIR} &&
    curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_model_checkpoints.tar.gz" -o emg2pose_model_checkpoints.tar.gz &&
    tar -xvzf emg2pose_model_checkpoints.tar.gz
    ''')

In [ ]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={DATA_DOWNLOAD_DIR}/emg2pose_model_checkpoints/regression_vemg2pose.ckpt"
    ]
)

In [ ]:
from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

In [ ]:
import torch

session = data 
start_idx = 0
stop_idx = len(session["emg"]) # eval on the whole session

session_window = session[start_idx:stop_idx]

# no_ik_failure is not a field so we slice separately
no_ik_failure_window = session.no_ik_failure[start_idx:stop_idx]

batch = {
    "emg": torch.Tensor([session_window["emg"].T]),  # BCT
    "joint_angles": torch.Tensor([session_window["joint_angles"].T]),  # BCT
    "no_ik_failure": torch.Tensor([no_ik_failure_window]),  # BT
}

preds, joint_angles, no_ik_failure = module.forward(batch)

# Algorithms that use the initial state for ground truth will do poorly
# when the first joint angles are missing!
if (joint_angles[:, 0] == 0).all():
    print(
        "Warning! Ground truth not available at first time step!"
    )

# BCT --> TC (as numpy)
preds = preds[0].T.detach().numpy()
joint_angles = joint_angles[0].T.detach().numpy()

print("Predictions Shape: " + str(preds.shape))
print("Ground Truth Shape " + str(joint_angles.shape))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Ensure alignment and scale

print(preds.shape)
print(joint_angles.shape)

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

print(preds_30hz.shape)
print(ground_truth_30hz.shape)

mask = no_ik_failure.cpu().numpy().squeeze()  # -> (T,)
mask_30hz = downsample(mask.astype(float), 2000, 30) > 0.5
mask_30hz = mask_30hz.squeeze()  # ensure (T,)

preds_30hz = preds_30hz[mask_30hz]
ground_truth_30hz = ground_truth_30hz[mask_30hz]

print(preds_30hz.shape)
print(ground_truth_30hz.shape)


In [ ]:
from emg2pose.evaluation import evaluate_predictions

evaluate_predictions(preds_30hz, ground_truth_30hz)

### 3b Small LSTM

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- load multiple sessions ---
X_all, y_all = [], []

for s in train:
    data = Emg2PoseSessionData(hdf5_path=s)
    X_all.append(data['emg'])
    y_all.append(data['joint_angles'])

X = np.concatenate(X_all, axis=0)
y = np.concatenate(y_all, axis=0)

# --- build sequences ---
seq_len = 50

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X, y, seq_len)

# split
split_idx = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

# tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

# model
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = LSTMModel(X.shape[1], 128, y.shape[1])

# train
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(10):
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(loss.item())

# eval
model.eval()
with torch.no_grad():
    y_pred = model(X_test)

In [ ]:
print(y.shape)

print(y_test.shape)
print(y_pred.shape)

In [ ]:
# --- inference on raw EMG sequence ---

def predict_stream(model, X, seq_len=50):
    model.eval()
    preds = []

    for i in range(len(X) - seq_len):
        window = X[i:i+seq_len]
        window = torch.tensor(window[None, ...], dtype=torch.float32)

        with torch.no_grad():
            pred = model(window).cpu().numpy()

        preds.append(pred[0])

    return np.array(preds)


# example usage:
data = Emg2PoseSessionData(hdf5_path=train[0])
X_raw = data['emg']

y_pred = predict_stream(model, X_raw, seq_len=50)

In [ ]:
ground_truth = data['joint_angles']

print(ground_truth.shape)

In [ ]:
preds_30hz = downsample(y_pred, 2000, 30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
ground_truth_30hz = downsample(ground_truth, 2000, 30)
visualization.get_plotly_animation_for_joint_angles(ground_truth_30hz[0:250], color="pink")

In [ ]:
evaluate_predictions(preds_30hz, ground_truth_30hz)

LSTM trained on user x achieved very good MAE when evaluated on that user.
Now we will consider this model trained on user x to see generalizabiltiy to other users (user y).

In [ ]:
user_y = sessions[30]
user_y

In [ ]:
# --- inference on raw EMG sequence ---

def predict_stream(model, X, seq_len=50):
    model.eval()
    preds = []

    for i in range(len(X) - seq_len):
        window = X[i:i+seq_len]
        window = torch.tensor(window[None, ...], dtype=torch.float32)

        with torch.no_grad():
            pred = model(window).cpu().numpy()

        preds.append(pred[0])

    return np.array(preds)


# example usage:
user_y_data = Emg2PoseSessionData(hdf5_path=user_y)
X_raw = user_y_data['emg']

y_pred = predict_stream(model, X_raw, seq_len=50)

In [ ]:
ground_truth = user_y_data['joint_angles']


In [ ]:
preds_30hz = downsample(y_pred, 2000, 30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
ground_truth_30hz = downsample(ground_truth, 2000, 30)
visualization.get_plotly_animation_for_joint_angles(ground_truth_30hz[0:250], color="pink")

In [ ]:
import importlib
import emg2pose.evaluation
importlib.reload(emg2pose.evaluation)

from emg2pose.evaluation import evaluate_predictions

In [ ]:
evaluate_predictions(preds_30hz, ground_truth_30hz)

### 3c Meta Model with Limited Training Data

In [ ]:
## Train your own model on a session

# import os
# import glob
# import shutil
# import subprocess
# import pandas as pd
# from pathlib import Path


# TMP_DIR = DATA_DOWNLOAD_DIR / "emg_single"

# # --- collect sessions ---
# print(f"Found {len(sessions)} sessions")

# # --- pick session ---
# idx = 0  # change this
# session_path = sessions[idx]
# filename = os.path.basename(session_path)

# print("Using:", filename)

# # --- prepare temp folder ---
# TMP_DIR.mkdir(exist_ok=True)

# # clear old contents
# for f in TMP_DIR.glob("*"):
#     if f.name.startswith("._"):
#         continue
#     f.unlink()

# # copy selected file
# shutil.copy(session_path, TMP_DIR / filename)

# # --- strip extension (CRITICAL) ---
# filename_no_ext = filename.replace(".hdf5", "")

# # --- create metadata.csv ---
# rows = []
# for split in ["train", "val", "test"]:
#     rows.append({
#         "session": filename_no_ext.split("-recording")[0],
#         "user": "debug_user",
#         "stage": "debug",
#         "start": 0,
#         "end": 1,
#         "side": "right",
#         "filename": filename_no_ext,
#         "moving_hand": "both",
#         "held_out_user": False,
#         "held_out_stage": False,
#         "split": split,
#         "generalization": "debug"
#     })

# df = pd.DataFrame(rows)
# df.to_csv(TMP_DIR / "metadata.csv", index=False)

# print("Created metadata.csv at:", TMP_DIR)

# # --- run training ---
# subprocess.run([
#     "python", "-m", "emg2pose.train",
#     "train=True",
#     "eval=True",
#     "experiment=tracking_vemg2pose",
#     "trainer.max_epochs=100",
#     "+callbacks.1.dirpath=/Volumes/Crucial X9/emg_checkpoints",
#     f"data_location={TMP_DIR}"
# ])

In [ ]:
# Grab checkpoint

from pathlib import Path

CKPT_DIR = Path("/Volumes/Crucial X9/emg_checkpoints")

ckpts = sorted(CKPT_DIR.glob("*.ckpt"), key=lambda x: x.stat().st_mtime)

best_ckpt = ckpts[-1]

print("Using checkpoint:", best_ckpt)

In [ ]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint=CKPT_DIR"
    ]
)

from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

In [ ]:
session = sessions[15]
data = Emg2PoseSessionData(hdf5_path=session)

In [ ]:
import torch

session = data 
start_idx = 0
stop_idx = len(session["emg"]) # eval on the whole session

session_window = session[start_idx:stop_idx]

# no_ik_failure is not a field so we slice separately
no_ik_failure_window = session.no_ik_failure[start_idx:stop_idx]

batch = {
    "emg": torch.Tensor([session_window["emg"].T]),  # BCT
    "joint_angles": torch.Tensor([session_window["joint_angles"].T]),  # BCT
    "no_ik_failure": torch.Tensor([no_ik_failure_window]),  # BT
}

preds, joint_angles, no_ik_failure = module.forward(batch)

# Algorithms that use the initial state for ground truth will do poorly
# when the first joint angles are missing!
if (joint_angles[:, 0] == 0).all():
    print(
        "Warning! Ground truth not available at first time step!"
    )

# BCT --> TC (as numpy)
preds = preds[0].T.detach().numpy()
joint_angles = joint_angles[0].T.detach().numpy()

print("Predictions Shape: " + str(preds.shape))
print("Ground Truth Shape " + str(joint_angles.shape))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Ensure alignment and scale

print(preds.shape)
print(joint_angles.shape)

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

print(preds_30hz.shape)
print(ground_truth_30hz.shape)

mask = no_ik_failure.cpu().numpy().squeeze()  # -> (T,)
mask_30hz = downsample(mask.astype(float), 2000, 30) > 0.5
mask_30hz = mask_30hz.squeeze()  # ensure (T,)

preds_30hz = preds_30hz[mask_30hz]
ground_truth_30hz = ground_truth_30hz[mask_30hz]

print(preds_30hz.shape)
print(ground_truth_30hz.shape)


In [ ]:
from emg2pose.evaluation import evaluate_predictions

evaluate_predictions(preds_30hz, ground_truth_30hz)